In [2]:
import sys
sys.path.append('../../Share/')
sys.path.append('../../V5 Self Supervised Learning')

import pandas as pd
import numpy as np
from sklearn.utils import resample
import matplotlib.pyplot as plt
import baseline, config, self_supervised_v1

import warnings
warnings.filterwarnings('ignore')

def evaluate_model(model, data, labels):
    return model.evaluate(data, labels, verbose=0)[1]


def down_sample(X, y):
    X = np.array(X)
    y = np.array(y)

    # 클래스별 최소 개수
    unique_classes, counts = np.unique(y, return_counts=True)
    min_count = counts.min()

    X_balanced, y_balanced = [], []

    for cls in unique_classes:
        idx = np.where(y == cls)[0]  # 해당 클래스 인덱스
        down_idx = resample(idx, replace=False, n_samples=min_count, random_state=42)
        X_balanced.append(X[down_idx])
        y_balanced.append(y[down_idx])

    # 합치기
    X_balanced = np.concatenate(X_balanced, axis=0)
    y_balanced = np.concatenate(y_balanced, axis=0)

    return X_balanced, y_balanced



trainer_Minjeong = baseline.ModelTrainer(config, subject="Minjeong")
trainer_Carlson = baseline.ModelTrainer(config, subject="Carlson")
trainer_Harold = baseline.ModelTrainer(config, subject="Harold")
trainer_Hunmin = baseline.ModelTrainer(config, subject="Hunmin")
trainer_Brian = baseline.ModelTrainer(config, subject="Brian")
trainer_Xianyu = baseline.ModelTrainer(config, subject="Xianyu")

In [1]:
def generate_pseudo_label(model, x):
    y_pred = model.predict(x, verbose=0)
    return np.argmax(y_pred, axis=1)

def online_update(model, new_sample, pseudo_label, X_test, y_test):
    #model.train_on_batch(new_sample, pseudo_label)

    history = model.fit(new_sample, pseudo_label, batch_size=256, epochs=50, validation_data=(X_test, y_test), verbose=0)
    return np.max(history.history['val_accuracy']), model

def evaluate_model(model, data, labels):
    return model.evaluate(data, labels, verbose=0)[1]

In [3]:
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix
import meta

trainers_unseen = {
    "H": trainer_Hunmin,
    "C": trainer_Carlson,
    "B": trainer_Brian,
    "X": trainer_Xianyu,
    "M": trainer_Minjeong
}
#Same_Session_Test_Acc, Next_Session_Test_Acc = [], []
#Unseen_subject_acc_dict = {"H": [], "C": [], "B": [], "X": [], "M": []}
n_classes = 6

In [ ]:
Harold_CM = []

In [30]:
model = load_model('./model_K10_H2.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Minjeong, len(config.Info_sub_M)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-15-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-08-01-v1/E9AD0E7DCC2B/
0.5583333373069763 0.46105918288230896
Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
0.5583333373069763 0.44548287987709045
Returning K-th session data: Exp_2025-08-02-v1/E9AD0E7DCC2B/
0.5083333253860474 0.4462616741657257
Returning K-th session data: Exp_2025-08-02-v2/E9AD0E7DCC2B/
0.5166666507720947 0.427570104598999
Returning K-th session data: Exp_2025-08-09-v1/E9AD0E7DCC2B/
0.5666666626930237 0.4462616741657257
Returning K-th session data: Exp_2025-08-09-v2/E9AD0E7DCC2B/
0.574999988079071 0.5218068361282349
Returning K-th session data: Exp_2025-08-10-v1/E9AD0E7DCC2B/
0.6166666746139526 0.5475077629089355
Returning K-th session data: Exp_2025-08-10-v2/E9AD0E7DCC2B/
0.6000000238418579 0.4961059093475342
Returning K-th session data: Exp_2025-08-11-v1/E9AD0E7DCC2B/
0.574999988079071 0.531931459903717
Returning K-th session data: Exp_2025-08-11-v2/E9A

In [31]:
model = load_model('./model_K10_H2.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Hunmin, len(config.Info_sub_H)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v3/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-05-27/E8331D05289A/
0.8333333134651184 0.526129961013794
Returning K-th session data: Exp_2025-06-18/E9AD0E7DCC2B/
0.7749999761581421 0.5317796468734741
Returning K-th session data: Exp_2025-06-20-v1/E9AD0E7DCC2B/
0.9083333611488342 0.5254237055778503
Returning K-th session data: Exp_2025-06-20-v2/E9AD0E7DCC2B/
0.8916666507720947 0.5353107452392578
Returning K-th session data: Exp_2025-06-20-v3/E9AD0E7DCC2B/
0.949999988079071 0.5656779408454895
Returning K-th session data: Exp_2025-06-20-v4/E9AD0E7DCC2B/
0.7666666507720947 0.48022598028182983
Returning K-th session data: Exp_2025-06-20-v5/E9AD0E7DCC2B/
0.8416666388511658 0.5353107452392578
Returning K-th session data: Exp_2025-06-20-v6/E9AD0E7DCC2B/
0.8083333373069763 0.5331920981407166
Returning K-th session data: Exp_2025-06-20-v7/E9AD0E7DCC2B/
0.875 0.5381355881690979
Returning K-th session data: Exp_2025-06-20-v8/E9AD0E7DCC2B/
0.95833

In [32]:
model = load_model('./model_K10_H2.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Xianyu, len(config.Info_sub_X)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-07-23-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-24-v1/E9AD0E7DCC2B/
0.574999988079071 0.4712389409542084
Returning K-th session data: Exp_2025-06-24-v2/E9AD0E7DCC2B/
0.8166666626930237 0.2654867172241211
Returning K-th session data: Exp_2025-06-26-v1/E9AD0E7DCC2B/
0.7250000238418579 0.43436577916145325
Returning K-th session data: Exp_2025-06-26-v2/E9AD0E7DCC2B/
0.5083333253860474 0.3023598790168762
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.574999988079071 0.33185839653015137
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.6916666626930237 0.2890855371952057
Returning K-th session data: Exp_2025-06-30-v1/FEFFF6FFF5FF/
0.5833333134651184 0.40265485644340515
Returning K-th session data: Exp_2025-06-30-v2/FEFFF6FFF5FF/
0.49166667461395264 0.35545721650123596
Returning K-th session data: Exp_2025-07-01-v1/E9AD0E7DCC2B/
0.40833333134651184 0.2337758094072342
Returning K-th session data: Exp_2025-07-01-

In [33]:
model = load_model('./model_K10_H2.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Brian, len(config.Info_sub_B)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.5 0.4086419641971588
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.7250000238418579 0.5646090507507324
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.8166666626930237 0.5390946269035339
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.7916666865348816 0.6008230447769165
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.875 0.7234567999839783
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.8833333253860474 0.7514403462409973
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.8583333492279053 0.7497942447662354
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.875 0.7576131820678711
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.8666666746139526 0.7946501970291138
Returning K-th session data: Exp_2025-07-17-v2/E9AD0E7DCC2B/
0.9166666865348816 0.8045267

In [34]:
model = load_model('./model_K10_H2.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Carlson, len(config.Info_sub_C)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-30-v1/E9AD0E7DCC2B/
0.5083333253860474 0.3589743673801422
Returning K-th session data: Exp_2025-06-30-v2/E9AD0E7DCC2B/
0.5333333611488342 0.2995951473712921
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.5833333134651184 0.37516868114471436
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.5333333611488342 0.3940620720386505
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.6000000238418579 0.5060728788375854
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.550000011920929 0.49325236678123474
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.550000011920929 0.5134952664375305
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.6499999761581421 0.5148447751998901
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.5833333134651184 0.4858299493789673
Returning K-th session data: Exp_2025-07-16-v2/E

# Hunmin

In [4]:
model = load_model('./model_K10_H.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Minjeong, len(config.Info_sub_M)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-15-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-08-01-v1/E9AD0E7DCC2B/
0.5416666865348816 0.4728682041168213
Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
0.6000000238418579 0.4674418568611145
Returning K-th session data: Exp_2025-08-02-v1/E9AD0E7DCC2B/
0.5583333373069763 0.5100775361061096
Returning K-th session data: Exp_2025-08-02-v2/E9AD0E7DCC2B/
0.5833333134651184 0.5294573903083801
Returning K-th session data: Exp_2025-08-09-v1/E9AD0E7DCC2B/
0.5833333134651184 0.5612403154373169
Returning K-th session data: Exp_2025-08-09-v2/E9AD0E7DCC2B/
0.6833333373069763 0.6007751822471619
Returning K-th session data: Exp_2025-08-10-v1/E9AD0E7DCC2B/
0.6166666746139526 0.6186046600341797
Returning K-th session data: Exp_2025-08-10-v2/E9AD0E7DCC2B/
0.625 0.6201550364494324
Returning K-th session data: Exp_2025-08-11-v1/E9AD0E7DCC2B/
0.6666666865348816 0.5906976461410522
Returning K-th session data: Exp_2025-08-11-v2/E9AD0E7DCC2B/


In [6]:
model = load_model('./model_K10_H.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Brian, len(config.Info_sub_B)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.6166666746139526 0.5352798104286194
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.824999988079071 0.48499593138694763
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.8083333373069763 0.6451743841171265
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.8333333134651184 0.6557177901268005
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.8500000238418579 0.7919707894325256
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.9166666865348816 0.8390105366706848
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.8999999761581421 0.8775344491004944
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.9333333373069763 0.9034874439239502
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.9333333373069763 0.8094079494476318
Returning K-th session data: Exp_2025-07-17-v2/E

In [7]:
model = load_model('./model_K10_H.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Xianyu, len(config.Info_sub_X)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-07-23-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-24-v1/E9AD0E7DCC2B/
0.6583333611488342 0.3644927442073822
Returning K-th session data: Exp_2025-06-24-v2/E9AD0E7DCC2B/
0.800000011920929 0.3224637806415558
Returning K-th session data: Exp_2025-06-26-v1/E9AD0E7DCC2B/
0.6833333373069763 0.4007246494293213
Returning K-th session data: Exp_2025-06-26-v2/E9AD0E7DCC2B/
0.5833333134651184 0.3007246255874634
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.699999988079071 0.38260868191719055
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.6666666865348816 0.28913044929504395
Returning K-th session data: Exp_2025-06-30-v1/FEFFF6FFF5FF/
0.5916666388511658 0.3065217435359955
Returning K-th session data: Exp_2025-06-30-v2/FEFFF6FFF5FF/
0.6083333492279053 0.4050724506378174
Returning K-th session data: Exp_2025-07-01-v1/E9AD0E7DCC2B/
0.5249999761581421 0.280434787273407
Returning K-th session data: Exp_2025-07-01-v2/E9

In [8]:
model = load_model('./model_K10_H.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Carlson, len(config.Info_sub_C)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-30-v1/E9AD0E7DCC2B/
0.4749999940395355 0.27952754497528076
Returning K-th session data: Exp_2025-06-30-v2/E9AD0E7DCC2B/
0.6416666507720947 0.3248031437397003
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.6583333611488342 0.39829397201538086
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.5833333134651184 0.41863515973091125
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.6666666865348816 0.5177165269851685
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.699999988079071 0.5485564470291138
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.675000011920929 0.5682414770126343
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.6499999761581421 0.5262467265129089
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.6083333492279053 0.5472440719604492
Returning K-th session data: Exp_2025-07-16-v2/

In [9]:
model = load_model('./model_K10_H.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Harold, len(config.Info_sub_H2)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.7083333134651184 0.6000000238418579
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.8500000238418579 0.5858024954795837
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.949999988079071 0.720370352268219
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.949999988079071 0.7580246925354004
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.9833333492279053 0.7592592835426331
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.949999988079071 0.8006172776222229
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.8999999761581421 0.790123462677002
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.9583333134651184 0.809876561164856
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.9416666626930237 0.8475308418273926
Returning K-th session data: Exp_2025-07-17-v2/E9AD0E7

# Xianyu

In [10]:
model = load_model('./model_K10_X.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Harold, len(config.Info_sub_H2)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.6833333373069763 0.5744157433509827
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.7250000238418579 0.5719557404518127
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.8583333492279053 0.5756457448005676
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.9166666865348816 0.5934809446334839
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.925000011920929 0.6506764888763428
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.925000011920929 0.739237368106842
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.8916666507720947 0.7490774989128113
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.9083333611488342 0.7509225010871887
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.8916666507720947 0.7595325708389282
Returning K-th session data: Exp_2025-07-17-v2/E9AD

In [11]:
model = load_model('./model_K10_X.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Hunmin, len(config.Info_sub_H)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v3/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-05-27/E8331D05289A/
0.6416666507720947 0.33893558382987976
Returning K-th session data: Exp_2025-06-18/E9AD0E7DCC2B/
0.7333333492279053 0.36344537138938904
Returning K-th session data: Exp_2025-06-20-v1/E9AD0E7DCC2B/
0.8999999761581421 0.424369752407074
Returning K-th session data: Exp_2025-06-20-v2/E9AD0E7DCC2B/
0.8916666507720947 0.4362744987010956
Returning K-th session data: Exp_2025-06-20-v3/E9AD0E7DCC2B/
0.925000011920929 0.46988794207572937
Returning K-th session data: Exp_2025-06-20-v4/E9AD0E7DCC2B/
0.7583333253860474 0.35364145040512085
Returning K-th session data: Exp_2025-06-20-v5/E9AD0E7DCC2B/
0.824999988079071 0.5042017102241516
Returning K-th session data: Exp_2025-06-20-v6/E9AD0E7DCC2B/
0.75 0.4978991448879242
Returning K-th session data: Exp_2025-06-20-v7/E9AD0E7DCC2B/
0.8500000238418579 0.5021008253097534
Returning K-th session data: Exp_2025-06-20-v8/E9AD0E7DCC2B/
0.9250

In [12]:
model = load_model('./model_K10_X.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Carlson, len(config.Info_sub_C)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-30-v1/E9AD0E7DCC2B/
0.46666666865348816 0.3339920938014984
Returning K-th session data: Exp_2025-06-30-v2/E9AD0E7DCC2B/
0.574999988079071 0.33662715554237366
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.5833333134651184 0.38801053166389465
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.5166666507720947 0.3893280625343323
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.5583333373069763 0.4947299063205719
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.5916666388511658 0.4308300316333771
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.6166666746139526 0.4743083119392395
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.6666666865348816 0.49736496806144714
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.5916666388511658 0.4888010621070862
Returning K-th session data: Exp_2025-07-16-v

In [13]:
model = load_model('./model_K10_X.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Brian, len(config.Info_sub_B)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.6583333611488342 0.5016260147094727
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.7666666507720947 0.5471544861793518
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.7749999761581421 0.5601626038551331
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.8166666626930237 0.5536585450172424
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.8166666626930237 0.6955284476280212
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.8833333253860474 0.7142276167869568
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.8500000238418579 0.7304878234863281
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.8916666507720947 0.7487804889678955
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.8999999761581421 0.7788617610931396
Returning K-th session data: Exp_2025-07-17-v2/E

In [14]:
model = load_model('./model_K10_X.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Minjeong, len(config.Info_sub_M)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-15-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-08-01-v1/E9AD0E7DCC2B/
0.5833333134651184 0.4042056202888489
Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
0.6166666746139526 0.4338006377220154
Returning K-th session data: Exp_2025-08-02-v1/E9AD0E7DCC2B/
0.6000000238418579 0.3901869058609009
Returning K-th session data: Exp_2025-08-02-v2/E9AD0E7DCC2B/
0.6416666507720947 0.427570104598999
Returning K-th session data: Exp_2025-08-09-v1/E9AD0E7DCC2B/
0.6583333611488342 0.43146416544914246
Returning K-th session data: Exp_2025-08-09-v2/E9AD0E7DCC2B/
0.7166666388511658 0.4267912805080414
Returning K-th session data: Exp_2025-08-10-v1/E9AD0E7DCC2B/
0.625 0.47274142503738403
Returning K-th session data: Exp_2025-08-10-v2/E9AD0E7DCC2B/
0.5916666388511658 0.4704049825668335
Returning K-th session data: Exp_2025-08-11-v1/E9AD0E7DCC2B/
0.6166666746139526 0.45093458890914917
Returning K-th session data: Exp_2025-08-11-v2/E9AD0E7DCC2B

# Brian

In [15]:
model = load_model('./model_K10_B.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Minjeong, len(config.Info_sub_M)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-15-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-08-01-v1/E9AD0E7DCC2B/
0.49166667461395264 0.3790697753429413
Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
0.5249999761581421 0.4635658860206604
Returning K-th session data: Exp_2025-08-02-v1/E9AD0E7DCC2B/
0.5583333373069763 0.45348837971687317
Returning K-th session data: Exp_2025-08-02-v2/E9AD0E7DCC2B/
0.5833333134651184 0.4937984347343445
Returning K-th session data: Exp_2025-08-09-v1/E9AD0E7DCC2B/
0.574999988079071 0.4961240291595459
Returning K-th session data: Exp_2025-08-09-v2/E9AD0E7DCC2B/
0.5833333134651184 0.4775193929672241
Returning K-th session data: Exp_2025-08-10-v1/E9AD0E7DCC2B/
0.5083333253860474 0.4310077428817749
Returning K-th session data: Exp_2025-08-10-v2/E9AD0E7DCC2B/
0.44999998807907104 0.43410852551460266
Returning K-th session data: Exp_2025-08-11-v1/E9AD0E7DCC2B/
0.5916666388511658 0.4658914804458618
Returning K-th session data: Exp_2025-08-11-v

In [16]:
model = load_model('./model_K10_B.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Hunmin, len(config.Info_sub_H)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v3/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-05-27/E8331D05289A/
0.699999988079071 0.4653954803943634
Returning K-th session data: Exp_2025-06-18/E9AD0E7DCC2B/
0.675000011920929 0.5042372941970825
Returning K-th session data: Exp_2025-06-20-v1/E9AD0E7DCC2B/
0.8999999761581421 0.4858756959438324
Returning K-th session data: Exp_2025-06-20-v2/E9AD0E7DCC2B/
0.8666666746139526 0.4781073331832886
Returning K-th session data: Exp_2025-06-20-v3/E9AD0E7DCC2B/
0.949999988079071 0.5007061958312988
Returning K-th session data: Exp_2025-06-20-v4/E9AD0E7DCC2B/
0.75 0.3820621371269226
Returning K-th session data: Exp_2025-06-20-v5/E9AD0E7DCC2B/
0.8333333134651184 0.4901129901409149
Returning K-th session data: Exp_2025-06-20-v6/E9AD0E7DCC2B/
0.699999988079071 0.5028248429298401
Returning K-th session data: Exp_2025-06-20-v7/E9AD0E7DCC2B/
0.75 0.5148305296897888
Returning K-th session data: Exp_2025-06-20-v8/E9AD0E7DCC2B/
0.9333333373069763 0.5211

In [17]:
model = load_model('./model_K10_B.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Harold, len(config.Info_sub_H2)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.8166666626930237 0.47469136118888855
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.9083333611488342 0.4117283821105957
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.8166666626930237 0.6283950805664062
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.8166666626930237 0.6833333373069763
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.824999988079071 0.7956790328025818
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.925000011920929 0.7944444417953491
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.8166666626930237 0.8493826985359192
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.8833333253860474 0.8395061492919922
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.875 0.8401234745979309
Returning K-th session data: Exp_2025-07-17-v2/E9AD0E7DCC2B/
0

In [18]:
model = load_model('./model_K10_B.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Xianyu, len(config.Info_sub_X)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-07-23-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-24-v1/E9AD0E7DCC2B/
0.5583333373069763 0.38325992226600647
Returning K-th session data: Exp_2025-06-24-v2/E9AD0E7DCC2B/
0.8416666388511658 0.29368576407432556
Returning K-th session data: Exp_2025-06-26-v1/E9AD0E7DCC2B/
0.5833333134651184 0.3612334728240967
Returning K-th session data: Exp_2025-06-26-v2/E9AD0E7DCC2B/
0.550000011920929 0.16813509166240692
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.49166667461395264 0.33406755328178406
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.625 0.3869309723377228
Returning K-th session data: Exp_2025-06-30-v1/FEFFF6FFF5FF/
0.6083333492279053 0.3972099721431732
Returning K-th session data: Exp_2025-06-30-v2/FEFFF6FFF5FF/
0.5916666388511658 0.37812042236328125
Returning K-th session data: Exp_2025-07-01-v1/E9AD0E7DCC2B/
0.4583333432674408 0.34875184297561646
Returning K-th session data: Exp_2025-07-01-v2/E9AD0E7D

In [19]:
model = load_model('./model_K10_B.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Carlson, len(config.Info_sub_C)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-30-v1/E9AD0E7DCC2B/
0.4833333194255829 0.3794466257095337
Returning K-th session data: Exp_2025-06-30-v2/E9AD0E7DCC2B/
0.42500001192092896 0.2555994689464569
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.574999988079071 0.45586296916007996
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.6916666626930237 0.5052700638771057
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.6833333373069763 0.49275362491607666
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.675000011920929 0.4591567814350128
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.75 0.4940711557865143
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.7333333492279053 0.5289855003356934
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.675000011920929 0.5342556238174438
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0

# Carlson

In [20]:
model = load_model('./model_K10_C.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Harold, len(config.Info_sub_H2)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.5249999761581421 0.47908979654312134
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.6000000238418579 0.6125461459159851
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.7833333611488342 0.6408364176750183
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.7083333134651184 0.6734317541122437
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.699999988079071 0.7269372940063477
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.875 0.7029520273208618
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.8666666746139526 0.7447724342346191
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.9583333134651184 0.7706027030944824
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.8916666507720947 0.7749077677726746
Returning K-th session data: Exp_2025-07-17-v2/E9AD0E7DCC2B/


In [21]:
model = load_model('./model_K10_C.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Hunmin, len(config.Info_sub_H)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v3/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-05-27/E8331D05289A/
0.6666666865348816 0.46061885356903076
Returning K-th session data: Exp_2025-06-18/E9AD0E7DCC2B/
0.6833333373069763 0.4676511883735657
Returning K-th session data: Exp_2025-06-20-v1/E9AD0E7DCC2B/
0.8083333373069763 0.4873417615890503
Returning K-th session data: Exp_2025-06-20-v2/E9AD0E7DCC2B/
0.8416666388511658 0.44936707615852356
Returning K-th session data: Exp_2025-06-20-v3/E9AD0E7DCC2B/
0.9083333611488342 0.5421940684318542
Returning K-th session data: Exp_2025-06-20-v4/E9AD0E7DCC2B/
0.7333333492279053 0.3734177350997925
Returning K-th session data: Exp_2025-06-20-v5/E9AD0E7DCC2B/
0.8416666388511658 0.5267229080200195
Returning K-th session data: Exp_2025-06-20-v6/E9AD0E7DCC2B/
0.6499999761581421 0.5393811464309692
Returning K-th session data: Exp_2025-06-20-v7/E9AD0E7DCC2B/
0.8333333134651184 0.5091420412063599
Returning K-th session data: Exp_2025-06-20-v8/E9AD0

In [22]:
model = load_model('./model_K10_C.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Brian, len(config.Info_sub_B)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.550000011920929 0.3178861737251282
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.6166666746139526 0.31016260385513306
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.7083333134651184 0.48170730471611023
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.75 0.4788617789745331
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.7666666507720947 0.6308943033218384
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.7583333253860474 0.677642285823822
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.8166666626930237 0.7182926535606384
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.7749999761581421 0.7658536434173584
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.8416666388511658 0.7504065036773682
Returning K-th session data: Exp_2025-07-17-v2/E9AD0E7DCC2B/
0

In [23]:
model = load_model('./model_K10_C.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Xianyu, len(config.Info_sub_X)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-07-23-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-24-v1/E9AD0E7DCC2B/
0.5166666507720947 0.3172514736652374
Returning K-th session data: Exp_2025-06-24-v2/E9AD0E7DCC2B/
0.6916666626930237 0.2492690086364746
Returning K-th session data: Exp_2025-06-26-v1/E9AD0E7DCC2B/
0.5916666388511658 0.36549708247184753
Returning K-th session data: Exp_2025-06-26-v2/E9AD0E7DCC2B/
0.5 0.2785087823867798
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.574999988079071 0.34283626079559326
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.625 0.3479532301425934
Returning K-th session data: Exp_2025-06-30-v1/FEFFF6FFF5FF/
0.5583333373069763 0.39912280440330505
Returning K-th session data: Exp_2025-06-30-v2/FEFFF6FFF5FF/
0.4749999940395355 0.2902046740055084
Returning K-th session data: Exp_2025-07-01-v1/E9AD0E7DCC2B/
0.32499998807907104 0.19298245012760162
Returning K-th session data: Exp_2025-07-01-v2/E9AD0E7DCC2B/
0.391666680

In [24]:
model = load_model('./model_K10_C.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Minjeong, len(config.Info_sub_M)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-15-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-08-01-v1/E9AD0E7DCC2B/
0.5249999761581421 0.4883720874786377
Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
0.6499999761581421 0.5441860556602478
Returning K-th session data: Exp_2025-08-02-v1/E9AD0E7DCC2B/
0.5583333373069763 0.5457364320755005
Returning K-th session data: Exp_2025-08-02-v2/E9AD0E7DCC2B/
0.5833333134651184 0.5193798542022705
Returning K-th session data: Exp_2025-08-09-v1/E9AD0E7DCC2B/
0.6166666746139526 0.5604650974273682
Returning K-th session data: Exp_2025-08-09-v2/E9AD0E7DCC2B/
0.6333333253860474 0.5821705460548401
Returning K-th session data: Exp_2025-08-10-v1/E9AD0E7DCC2B/
0.6333333253860474 0.551937997341156
Returning K-th session data: Exp_2025-08-10-v2/E9AD0E7DCC2B/
0.6583333611488342 0.5705426335334778
Returning K-th session data: Exp_2025-08-11-v1/E9AD0E7DCC2B/
0.6833333373069763 0.5767441987991333
Returning K-th session data: Exp_2025-08-11-v2/E9

# Minjeong

In [25]:
model = load_model('./model_K10_M.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Xianyu, len(config.Info_sub_X)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-07-23-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-24-v1/E9AD0E7DCC2B/
0.5666666626930237 0.3042212426662445
Returning K-th session data: Exp_2025-06-24-v2/E9AD0E7DCC2B/
0.75 0.235807865858078
Returning K-th session data: Exp_2025-06-26-v1/E9AD0E7DCC2B/
0.6583333611488342 0.37991267442703247
Returning K-th session data: Exp_2025-06-26-v2/E9AD0E7DCC2B/
0.5249999761581421 0.2154294103384018
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.6499999761581421 0.35735079646110535
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.7333333492279053 0.26128092408180237
Returning K-th session data: Exp_2025-06-30-v1/FEFFF6FFF5FF/
0.6166666746139526 0.3748180568218231
Returning K-th session data: Exp_2025-06-30-v2/FEFFF6FFF5FF/
0.5333333611488342 0.29985442757606506
Returning K-th session data: Exp_2025-07-01-v1/E9AD0E7DCC2B/
0.5249999761581421 0.30640465021133423
Returning K-th session data: Exp_2025-07-01-v2/E9AD0E7DCC2

In [26]:
model = load_model('./model_K10_M.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Hunmin, len(config.Info_sub_H)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v3/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-05-27/E8331D05289A/
0.800000011920929 0.46296295523643494
Returning K-th session data: Exp_2025-06-18/E9AD0E7DCC2B/
0.6666666865348816 0.5078347325325012
Returning K-th session data: Exp_2025-06-20-v1/E9AD0E7DCC2B/
0.8333333134651184 0.4772079885005951
Returning K-th session data: Exp_2025-06-20-v2/E9AD0E7DCC2B/
0.9166666865348816 0.47863247990608215
Returning K-th session data: Exp_2025-06-20-v3/E9AD0E7DCC2B/
0.9333333373069763 0.5398860573768616
Returning K-th session data: Exp_2025-06-20-v4/E9AD0E7DCC2B/
0.800000011920929 0.4216524362564087
Returning K-th session data: Exp_2025-06-20-v5/E9AD0E7DCC2B/
0.800000011920929 0.5534188151359558
Returning K-th session data: Exp_2025-06-20-v6/E9AD0E7DCC2B/
0.7833333611488342 0.5534188151359558
Returning K-th session data: Exp_2025-06-20-v7/E9AD0E7DCC2B/
0.875 0.5876068472862244
Returning K-th session data: Exp_2025-06-20-v8/E9AD0E7DCC2B/
0.90833

In [27]:
model = load_model('./model_K10_M.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Harold, len(config.Info_sub_H2)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.6666666865348816 0.6053921580314636
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.8583333492279053 0.6280637383460999
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.8666666746139526 0.6819853186607361
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.8833333253860474 0.7438725233078003
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.9750000238418579 0.7787989974021912
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.9666666388511658 0.8296568393707275
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.8333333134651184 0.8615196347236633
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.9333333373069763 0.873774528503418
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.8916666507720947 0.8897058963775635
Returning K-th session data: Exp_2025-07-17-v2/E9

In [28]:
model = load_model('./model_K10_M.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Brian, len(config.Info_sub_B)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-27-v1/E9AD0E7DCC2B/
0.5666666626930237 0.34878048300743103
Returning K-th session data: Exp_2025-06-27-v2/E9AD0E7DCC2B/
0.7333333492279053 0.4560975730419159
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.675000011920929 0.6134146451950073
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.8500000238418579 0.5735772252082825
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.75 0.6430894136428833
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.800000011920929 0.6390243768692017
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.75 0.6947154402732849
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0.8166666626930237 0.7004064917564392
Returning K-th session data: Exp_2025-07-17-v1/E9AD0E7DCC2B/
0.8666666746139526 0.7109755873680115
Returning K-th session data: Exp_2025-07-17-v2/E9AD0E7DCC2B/
0.92500001192092

In [29]:
model = load_model('./model_K10_M.h5')
model.compile(optimizer=Adam(learning_rate=1e-2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainer_sub, final_session = trainer_Carlson, len(config.Info_sub_C)
X_TEST, y_TEST, _, _ = trainer_sub.return_K_th_data_only(K=final_session-1, train_ratio=0.99)
X_TEST, y_TEST = down_sample(X_TEST, y_TEST)
Acc_lst, Acc_after_self_learning = [],[]

for k in range(10):
    unseen_X, unseen_y, X_few_shot, y_few_shot = trainer_sub.return_K_th_data_only(K=k, train_ratio=0.8)
    X_train, y_train = down_sample(unseen_X, unseen_y)
    X_test, y_test = down_sample(X_few_shot, y_few_shot)

    Meta = meta.MetaLearner(input_model=model, N_way=6, input_shape=X_train.shape[1:],
        meta_iters=5,                              # Number of meta-training loops
        meta_step_size=1                          # Reptile meta step
    )
    acc = Meta.train(X_train, y_train, X_TEST, y_TEST, meta.get_data_Meta, N_way=6, K_shot=20)

    pseudo_labels = generate_pseudo_label(model, X_train)
    acc2, model = online_update(model, X_train, pseudo_labels, X_TEST, y_TEST)
    Acc_after_self_learning.append(acc2)
    print(acc, acc2)
    Acc_lst.append(acc)

Returning K-th session data: Exp_2025-08-01-v2/E9AD0E7DCC2B/
Returning K-th session data: Exp_2025-06-30-v1/E9AD0E7DCC2B/
0.42500001192092896 0.33596837520599365
Returning K-th session data: Exp_2025-06-30-v2/E9AD0E7DCC2B/
0.6083333492279053 0.5862977504730225
Returning K-th session data: Exp_2025-07-09-v1/E9AD0E7DCC2B/
0.6416666507720947 0.647562563419342
Returning K-th session data: Exp_2025-07-09-v2/E9AD0E7DCC2B/
0.6666666865348816 0.6277997493743896
Returning K-th session data: Exp_2025-07-10-v1/E9AD0E7DCC2B/
0.7166666388511658 0.6660078763961792
Returning K-th session data: Exp_2025-07-10-v2/E9AD0E7DCC2B/
0.75 0.6798418760299683
Returning K-th session data: Exp_2025-07-11-v1/E9AD0E7DCC2B/
0.6833333373069763 0.6409749388694763
Returning K-th session data: Exp_2025-07-11-v2/E9AD0E7DCC2B/
0.699999988079071 0.6501976251602173
Returning K-th session data: Exp_2025-07-16-v1/E9AD0E7DCC2B/
0.6666666865348816 0.6528326869010925
Returning K-th session data: Exp_2025-07-16-v2/E9AD0E7DCC2B/
0